# Fase 3 · M01: Validación y Limpieza

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M01 — Validación |

---

## 🎯 Qué hace

Valida y limpia el dataset procesado eliminando duplicados, corrigiendo tipos y documentando anomalías. Primera capa de calidad antes del feature engineering.

## 📋 Requisitos

- `data/02_processed/df_alumno.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/03_features/df_alumno_limpio.parquet` | Dataset validado y limpio (42 cols + gap_calculado, orden_preferencia int) |
| `docs/html/fase3/m01_validacion.html` | Informe HTML |

## 🔄 Flujo

```
data/02_processed/df_alumno.parquet
    ↓ Detección de duplicados y anomalías
    ↓ Corrección de tipos
    ↓ Documentación de cambios
    → data/03_features/df_alumno_limpio.parquet + HTML
```

## ➡️ Siguiente

`f3_m02_agregacion.ipynb` — agregación por expediente académico


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import RUTA_PROCESSED, RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64, COLORES
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_pagina_desde_fichero

RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_FASE3_HTML])

ARCHIVO_ENTRADA = 'df_alumno.parquet'
ARCHIVO_SALIDA = 'df_alumno_limpio.parquet'

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('F3-M01: VALIDACIÓN Y LIMPIEZA')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / ARCHIVO_ENTRADA)
fmt = formato_numero_es

n_filas_antes = len(df)
n_cols_antes = len(df.columns)
cols_antes = set(df.columns)
duplicados_antes = df.duplicated().sum()

cols_nulos_posibles = ['via_acceso', 'forma_de_pago', 'numero_pagos', 'sexo']
cols_nulos_check = [c for c in cols_nulos_posibles if c in df.columns]

metricas_antes = {'filas': n_filas_antes, 'columnas': n_cols_antes, 'duplicados': duplicados_antes}
for col in cols_nulos_check:
    metricas_antes[f'pct_nulos_{col}'] = df[col].isna().mean() * 100

print(f'Entrada: {ARCHIVO_ENTRADA}')
print(f'Cargado: {fmt(n_filas_antes)} filas x {n_cols_antes} columnas')
print(f'Columnas: {sorted(df.columns.tolist())}')

F3-M01: VALIDACIÓN Y LIMPIEZA


Entrada: df_alumno.parquet
Cargado: 109.568 filas x 37 columnas
Columnas: ['cred_matriculados', 'cred_superados', 'cred_titulacion', 'cupo', 'curso_aca', 'curso_aca_fin', 'curso_aca_ini', 'egresado', 'exp_tit_id', 'fecha_nacimiento', 'forma_de_pago', 'id_pais', 'media_curso', 'media_titulacion_alumno', 'media_titulacion_curso', 'nombre_beca', 'nombre_trabajo', 'nota', 'nota_acceso', 'nota_selectividad', 'nuevo', 'numero_pagos', 'orden_preferencia', 'pais_domicilio', 'pais_nombre', 'per_id_ficticio', 'poblacion', 'provincia', 'rama', 'seguro', 'sexo', 'tiene_beca', 'titulacion', 'universidad_origen', 'via_acceso', 'via_acceso_exp', 'vive_fuera']


In [3]:
# ============================================================================
# CELDA 3: CORRECCIONES DE AUDITORÍA (P1-P7)
# ============================================================================

print('\n' + '-' * 60)
print('CORRECCIONES DE AUDITORÍA (P1-P7)')
print('-' * 60)

transformaciones = []

# --- P1: Sexo int → string (1/2 → Hombre/Mujer) ---
if 'sexo' in df.columns:
    valores = df['sexo'].dropna().unique()
    if set(valores).issubset({1, 2, '1', '2', 1.0, 2.0}):
        df['sexo'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer', '1': 'Hombre', '2': 'Mujer', 1.0: 'Hombre', 2.0: 'Mujer'})
        transformaciones.append(('sexo', 'P1: Codificado como int (1/2)', 'Mapear a Hombre/Mujer'))
        print('P1 ✓ sexo: traducido 1/2 → Hombre/Mujer')
    else:
        print(f'P1 ⏭️ sexo: ya es string ({valores[:3]}...)')

# --- P2: Renombrar 'nombre' → 'via_acceso_exp' (si existe) ---
if 'nombre' in df.columns:
    df = df.rename(columns={'nombre': 'via_acceso_exp'})
    df['via_acceso_exp'] = df['via_acceso_exp'].astype(str).str.strip()
    transformaciones.append(('via_acceso_exp', 'P2: Campo "nombre" es vía de acceso', 'Renombrar + limpiar'))
    print('P2 ✓ nombre → via_acceso_exp (renombrada + limpiada)')
else:
    print('P2 ⏭️ "nombre" no existe en el dataset')

# --- P3: Renombrar 'nota' → 'nota_expediente' ---
if 'nota' in df.columns:
    df = df.rename(columns={'nota': 'nota_expediente'})
    transformaciones.append(('nota_expediente', 'P3: Columna ambigua "nota"', 'Renombrar a nota_expediente'))
    print('P3 ✓ nota → nota_expediente (renombrada)')
elif 'nota_expediente' in df.columns:
    print('P3 ⏭️ ya existe nota_expediente')

# --- P5: via_acceso duplicada ---
if 'via_acceso' in df.columns and 'via_acceso_exp' in df.columns:
    print('P5 ⚠️ via_acceso y via_acceso_exp coexisten (M02 decide cuál agregar)')
elif 'via_acceso' in df.columns:
    print('P5 ✓ Solo existe via_acceso (ok)')

# --- P6: Eliminar id_pais (redundante con pais_nombre) ---
if 'id_pais' in df.columns:
    df = df.drop(columns=['id_pais'])
    transformaciones.append(('id_pais', 'P6: Redundante con pais_nombre', 'Eliminada'))
    print('P6 ✓ id_pais eliminada (redundante)')
else:
    print('P6 ⏭️ id_pais no existe')

# --- P7: Eliminar pais_domicilio (99.7% España) ---
if 'pais_domicilio' in df.columns:
    df = df.drop(columns=['pais_domicilio'])
    transformaciones.append(('pais_domicilio', 'P7: 99.7% España', 'Eliminada'))
    print('P7 ✓ pais_domicilio eliminada (99.7% España)')
else:
    print('P7 ⏭️ pais_domicilio no existe')

# --- Nulos via_acceso ---
if 'via_acceso' in df.columns:
    n_nulos = df['via_acceso'].isna().sum()
    if n_nulos > 0:
        df['via_acceso'] = df['via_acceso'].fillna('Sin datos')
        transformaciones.append(('via_acceso', 'Nulos en vía acceso', 'Rellenar con "Sin datos"'))
    print(f'via_acceso: {fmt(n_nulos)} nulos')

# --- P8: orden_preferencia — NaN → 0, float → int ---
# NaN significa que el alumno entró sin proceso de preinscripción
# (traslado, mayores de 25, otras vías especiales).
# Para ellos no existe orden de preferencia porque no eligieron:
# solo tenían disponible la carrera a la que accedieron.
# Se asigna 0 como valor especial: "entró sin elección".
# Los valores 1-20 indican la posición de preferencia en preinscripción.
if 'orden_preferencia' in df.columns:
    n_nulos_op = df['orden_preferencia'].isna().sum()
    df['orden_preferencia'] = df['orden_preferencia'].fillna(0).astype(int)
    transformaciones.append((
        'orden_preferencia',
        f'P8: {n_nulos_op:,} NaN — entró sin preinscripción (traslado, mayores 25, etc.)',
        'Rellenar con 0 (sin elección) + convertir a int'
    ))
    print(f'P8 ✓ orden_preferencia: {n_nulos_op:,} NaN → 0, convertido a int')
else:
    print('P8 ⏭️ orden_preferencia no existe en el dataset')

print(f'\n✅ {len(transformaciones)} correcciones aplicadas')


------------------------------------------------------------
CORRECCIONES DE AUDITORÍA (P1-P7)
------------------------------------------------------------
P1 ✓ sexo: traducido 1/2 → Hombre/Mujer
P2 ⏭️ "nombre" no existe en el dataset


P3 ✓ nota → nota_expediente (renombrada)
P5 ⚠️ via_acceso y via_acceso_exp coexisten (M02 decide cuál agregar)
P6 ✓ id_pais eliminada (redundante)
P7 ✓ pais_domicilio eliminada (99.7% España)
via_acceso: 0 nulos
P8 ✓ orden_preferencia: 13,835 NaN → 0, convertido a int

✅ 5 correcciones aplicadas


In [4]:
# ============================================================================
# CELDA 4: VARIABLES PRE-AGREGACIÓN
# ============================================================================
# Estas variables NECESITAN granularidad alumno×curso para calcularse.
# Por eso van en M01 y no en M02/M03.
# M02 las usa en la agregación (ej: mean(cred_superados_anio) por expediente).
# ============================================================================

print('\n' + '-' * 60)
print('VARIABLES PRE-AGREGACIÓN')
print('-' * 60)

# 4.1 Año de entrada corregido (min curso_aca del expediente)
if 'curso_aca' in df.columns:
    df['anio_entrada_corregido'] = df.groupby(['per_id_ficticio', 'exp_tit_id'])['curso_aca'].transform('min')
    transformaciones.append(('anio_entrada_corregido', 'Discrepancia en año entrada', 'min(curso_aca) del expediente'))
    print(f'✅ anio_entrada_corregido: rango {df["anio_entrada_corregido"].min()}-{df["anio_entrada_corregido"].max()}')

# 4.2 Año nacimiento + Edad calculada
if 'fecha_nacimiento' in df.columns:
    df['anio_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'], errors='coerce').dt.year
    df['edad_entrada_calc'] = df['anio_entrada_corregido'] - df['anio_nacimiento']
    transformaciones.append(('edad_entrada_calc', 'Edad calculada', 'anio_entrada - anio_nacimiento'))
    print(f'✅ edad_entrada_calc: rango {df["edad_entrada_calc"].min():.0f}-{df["edad_entrada_calc"].max():.0f}, media {df["edad_entrada_calc"].mean():.1f}')
else:
    print('⚠️ fecha_nacimiento no encontrada → edad_entrada_calc no creada')

# 4.3 Créditos superados por año (cred_superados es acumulativo → calcular diferencia)
if 'cred_superados' in df.columns:
    df = df.sort_values(['per_id_ficticio', 'exp_tit_id', 'curso_aca'])
    df['cred_superados_prev'] = df.groupby(['per_id_ficticio', 'exp_tit_id'])['cred_superados'].shift(1)
    df['cred_superados_anio'] = df['cred_superados'] - df['cred_superados_prev']
    # Primer año: superados = el valor directo
    mask_primer = df['cred_superados_prev'].isna()
    df.loc[mask_primer, 'cred_superados_anio'] = df.loc[mask_primer, 'cred_superados']
    df = df.drop(columns=['cred_superados_prev'])
    transformaciones.append(('cred_superados_anio', 'cred_superados es acumulativo', 'Diferencia entre cursos consecutivos'))
    print(f'✅ cred_superados_anio: media {df["cred_superados_anio"].mean():.1f}, rango {df["cred_superados_anio"].min():.0f}-{df["cred_superados_anio"].max():.0f}')

print(f'\n📊 Columnas tras pre-agregación: {len(df.columns)}')


------------------------------------------------------------
VARIABLES PRE-AGREGACIÓN
------------------------------------------------------------
✅ anio_entrada_corregido: rango 2010-2020
✅ edad_entrada_calc: rango 5-68, media 20.4


✅ cred_superados_anio: media 45.0, rango -18-252

📊 Columnas tras pre-agregación: 39


In [5]:
# ============================================================================
# CELDA 5: INDICADORES DE AUDITORÍA
# ============================================================================

print('\n' + '-' * 60)
print('INDICADORES DE AUDITORÍA')
print('-' * 60)

indicadores_resumen = []

# Indicador 1: Edad inusual (<15 años al entrar)
if 'edad_entrada_calc' in df.columns:
    df['indicador_edad_inusual'] = (df['edad_entrada_calc'] < 16).astype(int)
    n = df['indicador_edad_inusual'].sum()
    pct = n / len(df) * 100
    indicadores_resumen.append(('indicador_edad_inusual', '<15 años al entrar', n, pct))
    print(f'indicador_edad_inusual: {fmt(n)} ({pct:.2f}%)')

# Indicador 2: Interrupción (gaps en cursos académicos)
if 'curso_aca' in df.columns:
    def tiene_gaps(g):
        cursos = sorted(g['curso_aca'].unique())
        if len(cursos) < 2: return False
        for i in range(len(cursos) - 1):
            if cursos[i+1] - cursos[i] > 1: return True
        return False
    gaps = df.groupby(['per_id_ficticio', 'exp_tit_id']).apply(tiene_gaps, include_groups=False).reset_index(name='indicador_interrupcion')
    df = df.merge(gaps, on=['per_id_ficticio', 'exp_tit_id'], how='left')
    df['indicador_interrupcion'] = df['indicador_interrupcion'].astype(int)
    n = df['indicador_interrupcion'].sum()
    pct = n / len(df) * 100
    indicadores_resumen.append(('indicador_interrupcion', 'Gaps en trayectoria', n, pct))
    print(f'indicador_interrupcion: {fmt(n)} ({pct:.2f}%)')

    # anios_gap: cuántos años de gap tiene el expediente
    # Fórmula: (curso_ultimo - curso_inicio + 1) - n_cursos_reales
    # Ejemplo per_id=3360: (2019-2014+1) - 5 = 1 año de gap (faltó 2018)
    # Ejemplo per_id=14594: (2019-2013+1) - 5 = 2 años de gap (faltaron 2014 y 2017)
    # Valor 0 = sin gaps (trayectoria continua)
    # Se calcula a nivel expediente y se propaga a todas las filas del alumno.
    anos_posibles = df.groupby(['per_id_ficticio', 'exp_tit_id'])['curso_aca'].transform(lambda x: x.max() - x.min() + 1)
    n_cursos_reales = df.groupby(['per_id_ficticio', 'exp_tit_id'])['curso_aca'].transform('nunique')
    df['anios_gap'] = (anos_posibles - n_cursos_reales).astype(int)
    n_gap = (df['anios_gap'] > 0).sum()
    print(f'anios_gap: {fmt(n_gap)} registros con gap > 0, media gap = {df[df["anios_gap"]>0]["anios_gap"].mean():.1f} años')

# Indicador 3: Sin notas registradas
if 'media_curso' in df.columns:
    df['indicador_sin_notas'] = (df['media_curso'].isna() | (df['media_curso'] == 0)).astype(int)
    n = df['indicador_sin_notas'].sum()
    pct = n / len(df) * 100
    indicadores_resumen.append(('indicador_sin_notas', 'Sin notas registradas', n, pct))
    print(f'indicador_sin_notas: {fmt(n)} ({pct:.2f}%)')

print(f'\nTotal indicadores: {len(indicadores_resumen)}')


------------------------------------------------------------
INDICADORES DE AUDITORÍA
------------------------------------------------------------
indicador_edad_inusual: 1 (0.00%)


indicador_interrupcion: 5.191 (4.74%)


anios_gap: 5.191 registros con gap > 0, media gap = 1.5 años
indicador_sin_notas: 7.915 (7.22%)

Total indicadores: 3


In [6]:
# ============================================================================
# CELDA 6: RESUMEN ANTES/DESPUÉS
# ============================================================================

n_filas_despues = len(df)
n_cols_despues = len(df.columns)
cols_nuevas = sorted(set(df.columns) - cols_antes)
duplicados_despues = df.duplicated().sum()

metricas_despues = {'filas': n_filas_despues, 'columnas': n_cols_despues, 'duplicados': duplicados_despues}
for col in cols_nulos_check:
    if col in df.columns:
        metricas_despues[f'pct_nulos_{col}'] = df[col].isna().mean() * 100

print(f'\nResultado: {fmt(n_filas_despues)} filas x {n_cols_despues} columnas')
print(f'Columnas nuevas ({len(cols_nuevas)}): {cols_nuevas}')
print(f'\nTipos de datos:')
for dtype, count in df.dtypes.value_counts().items():
    print(f'  {dtype}: {count}')


Resultado: 109.568 filas x 43 columnas


Columnas nuevas (9): ['anio_entrada_corregido', 'anio_nacimiento', 'anios_gap', 'cred_superados_anio', 'edad_entrada_calc', 'indicador_edad_inusual', 'indicador_interrupcion', 'indicador_sin_notas', 'nota_expediente']

Tipos de datos:
  object: 16
  int64: 11
  float64: 11
  bool: 2
  Int64: 1
  datetime64[ns]: 1
  int32: 1


In [7]:
# ============================================================================
# CELDA 7: GRÁFICOS
# ============================================================================

print('\nGRÁFICOS')
print('-' * 40)

graficos_base64 = {}

# Gráfico 1: Impacto en Nulos (antes vs después)
if cols_nulos_check:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    vals_antes = [metricas_antes.get(f'pct_nulos_{c}', 0) for c in cols_nulos_check]
    vals_despues = [metricas_despues.get(f'pct_nulos_{c}', 0) for c in cols_nulos_check]
    axes[0].bar(cols_nulos_check, vals_antes, color=COLORES['danger'])
    axes[0].set_title('Antes', fontweight='bold')
    axes[0].set_ylabel('% Nulos')
    axes[0].tick_params(axis='x', rotation=45)
    colors_d = [COLORES['success'] if v == 0 else COLORES['warning'] for v in vals_despues]
    axes[1].bar(cols_nulos_check, vals_despues, color=colors_d)
    axes[1].set_title('Después', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    plt.suptitle('Impacto en Nulos', fontweight='bold', fontsize=12)
    plt.tight_layout()
    graficos_base64['nulos'] = figura_a_base64(fig)
    plt.close()

# Gráfico 2: Distribución Edad al Entrar
if 'edad_entrada_calc' in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df['edad_entrada_calc'].dropna(), bins=30, color=COLORES['primary'], edgecolor='white')
    ax.axvline(15, color=COLORES['danger'], linestyle='--', linewidth=2, label='Límite 15 años')
    ax.set_xlabel('Edad al Entrar')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Edad al Entrar', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    graficos_base64['edad'] = figura_a_base64(fig)
    plt.close()

# Gráfico 3: Distribución Nota de Acceso
if 'nota_acceso' in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df['nota_acceso'].dropna(), bins=30, color=COLORES['success'], edgecolor='white')
    ax.set_xlabel('Nota de Acceso')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Nota de Acceso', fontweight='bold')
    plt.tight_layout()
    graficos_base64['nota'] = figura_a_base64(fig)
    plt.close()

# Gráfico 4: Indicadores de Auditoría
if indicadores_resumen:
    fig, ax = plt.subplots(figsize=(8, 4))
    nombres = [r[0].replace('indicador_', '') for r in indicadores_resumen]
    valores = [r[2] for r in indicadores_resumen]
    colores_ind = [COLORES['danger'], COLORES['warning'], COLORES['primary'], COLORES.get('gray','#a0aec0')][:len(nombres)]
    ax.barh(nombres, valores, color=colores_ind)
    ax.set_xlabel('Número de Registros')
    ax.set_title('Indicadores de Auditoría', fontweight='bold')
    for i, v in enumerate(valores):
        ax.text(v + max(valores)*0.01, i, fmt(v), va='center', fontsize=9)
    plt.tight_layout()
    graficos_base64['indicadores'] = figura_a_base64(fig)
    plt.close()

print(f'{len(graficos_base64)} gráficos generados')


GRÁFICOS
----------------------------------------


4 gráficos generados


In [8]:
# ============================================================================
# CELDA 8: GUARDAR DATASET
# ============================================================================

ruta_salida = RUTA_FEATURES / ARCHIVO_SALIDA
df.to_parquet(ruta_salida, index=False)
print(f'💾 Guardado: {ARCHIVO_SALIDA} ({len(df.columns)} columnas)')
print(f'   Ruta: {ruta_salida}')

💾 Guardado: df_alumno_limpio.parquet (43 columnas)
   Ruta: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\df_alumno_limpio.parquet


In [9]:
# ============================================================================
# CELDA 8b: GUARDAR DATASET LONGITUDINAL
# ============================================================================
# df_alumno_limpio tiene 1 fila por alumno×año — es el histórico completo.
# Aquí se guarda una versión seleccionada con los campos más relevantes
# para análisis de trayectoria, supervivencia y futuros proyectos.
#
# Incluye cred_superados_anio (créditos reales ese año, NO acumulado),
# gap_calculado (años de interrupción) y orden_preferencia ya limpia.
#
# Fichero: data/03_features/df_longitudinal_trayectoria.parquet
# Uso previsto: análisis de supervivencia (Kaplan-Meier), series temporales,
#               seguimiento del progreso anual por alumno.
# ============================================================================

print('\n' + '-' * 60)
print('GUARDAR DATASET LONGITUDINAL')
print('-' * 60)

cols_longitudinal = [
    # Identificadores
    'per_id_ficticio', 'exp_tit_id',
    # Temporal
    'curso_aca', 'anio_entrada_corregido',
    # Créditos año a año (cred_superados_anio = créditos reales ese año, NO acumulado)
    'cred_matriculados', 'cred_superados', 'cred_superados_anio',
    # Rendimiento
    'media_curso', 'egresado',
    # Titulación
    'titulacion', 'rama', 'cred_titulacion',
    # Demográfico
    'sexo', 'edad_entrada_calc',
    # Acceso (orden_preferencia: 0=sin preinscripción, 1-20=posición elegida)
    'via_acceso', 'orden_preferencia', 'cupo',
    # Beca y laboral
    'tiene_beca', 'nombre_trabajo',
    # Indicadores de trayectoria
    'indicador_edad_inusual', 'indicador_interrupcion',
    'gap_calculado',  # equivalente a anios_gap en dataset de modelado
    'indicador_sin_notas',
    # Contexto grupo
    'media_titulacion_curso',
]

cols_exist = [c for c in cols_longitudinal if c in df.columns]
cols_missing = [c for c in cols_longitudinal if c not in df.columns]
if cols_missing:
    print(f'⚠️  Columnas no encontradas: {cols_missing}')

df_long = df[cols_exist].sort_values(
    ['per_id_ficticio', 'exp_tit_id', 'curso_aca']
).reset_index(drop=True)

ruta_long = RUTA_FEATURES / 'df_longitudinal_trayectoria.parquet'
df_long.to_parquet(ruta_long, index=False)

print(f'💾 Guardado: {ruta_long.name}')
print(f'   Filas:    {len(df_long):,} (1 por alumno×año)')
print(f'   Columnas: {len(df_long.columns)}')
print(f'   Alumnos:  {df_long["per_id_ficticio"].nunique():,}')
print(f'\n   Campos clave:')
print(f'   - cred_superados_anio: créditos reales ese año (no acumulado)')
print(f'   - gap_calculado: años de interrupción (0=continuo)')
print(f'   - orden_preferencia: 0=sin preinscripción, 1-20=posición elegida')



------------------------------------------------------------
GUARDAR DATASET LONGITUDINAL
------------------------------------------------------------
⚠️  Columnas no encontradas: ['gap_calculado']
💾 Guardado: df_longitudinal_trayectoria.parquet
   Filas:    109,568 (1 por alumno×año)
   Columnas: 23
   Alumnos:  30,872

   Campos clave:
   - cred_superados_anio: créditos reales ese año (no acumulado)
   - gap_calculado: años de interrupción (0=continuo)
   - orden_preferencia: 0=sin preinscripción, 1-20=posición elegida


In [10]:
# ============================================================================
# CELDA 9: GENERAR HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase3', modulo_activo='m01')

KPIS = [
    {'valor': fmt(n_filas_despues), 'titulo': 'Registros', 'color': COLORES['primary']},
    {'valor': str(n_cols_despues), 'titulo': 'Columnas', 'color': COLORES['success']},
    {'valor': f'+{len(cols_nuevas)}', 'titulo': 'Nuevas', 'color': COLORES['warning']},
    {'valor': str(len(indicadores_resumen)), 'titulo': 'Indicadores', 'color': COLORES['danger']},
]

# Tabla correcciones P1-P7
tabla_correcciones = '''<table style="width:100%;border-collapse:collapse;">
<tr style="background:#38a169;color:white;"><th style="padding:10px;">Columna</th><th>Problema</th><th>Solución</th></tr>\n'''
for t in transformaciones:
    tabla_correcciones += f'<tr><td><code>{t[0]}</code></td><td>{t[1]}</td><td>{t[2]}</td></tr>\n'
tabla_correcciones += '</table>'
s1 = generar_seccion_html('🔧 Correcciones y Variables Creadas', tabla_correcciones)

# Gráficos
graficos_html = '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:20px;">'
for key in ['nulos', 'edad', 'nota', 'indicadores']:
    if key in graficos_base64:
        graficos_html += f'<div style="text-align:center;"><img src="data:image/png;base64,{graficos_base64[key]}" style="max-width:100%;border-radius:8px;"></div>'
graficos_html += '</div>'
s2 = generar_seccion_html('📊 Visualizaciones', graficos_html)

html = render_pagina_desde_fichero(
    'f3_m01_validacion.ipynb',
    generar_kpis_html(KPIS) + s1 + s2,
    carpeta_notebook='fase3_features'
)
guardar_html(html, RUTA_FASE3_HTML / 'm01_validacion.html')
print('HTML generado')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m01_validacion.html
HTML generado


In [11]:
# ============================================================================
# CELDA 10: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M01 COMPLETADO')
print('=' * 60)
print(f'\nEntrada: {ARCHIVO_ENTRADA} ({fmt(n_filas_antes)} filas × {n_cols_antes} cols)')
print(f'Salida: {ARCHIVO_SALIDA} ({fmt(n_filas_despues)} filas × {n_cols_despues} cols)')
print(f'\nCorrecciones P1-P7: {len([t for t in transformaciones if t[1].startswith("P")])}')
print(f'Variables pre-agregación: anio_entrada_corregido, anio_nacimiento, edad_entrada_calc, cred_superados_anio, anios_gap')
print(f'Indicadores: {len(indicadores_resumen)}')
print(f'\n🎯 Siguiente: f3_m02_agregacion.ipynb (GROUP BY expediente)')
print('=' * 60)


✅ F3-M01 COMPLETADO

Entrada: df_alumno.parquet (109.568 filas × 37 cols)
Salida: df_alumno_limpio.parquet (109.568 filas × 43 cols)

Correcciones P1-P7: 5
Variables pre-agregación: anio_entrada_corregido, anio_nacimiento, edad_entrada_calc, cred_superados_anio, anios_gap
Indicadores: 3

🎯 Siguiente: f3_m02_agregacion.ipynb (GROUP BY expediente)
